# Hidden Markov Models
A Hidden Markov Model (HMM) is a class of probabilistic graphical model (PGM) with $2T$ variables $X_1, \dots, X_T$ and $Y_1 \dots, Y_T$.
We can represent a HMM with the following PGM:

![hmm](hmm.png)

The variables $X_t$ corresponds to the state at timestep $t$ and $Y_t$ corresponds to the observation of the state at timestep $t$.
What's important here is the assumption that the state of the current timestep depends only on the previous timestep (i.e. $\mathbb{P}(X_t | X_1, \dots, X_{t - 1}) = \mathbb{P}(X_t | X_{t - 1})$).

That means, we can write the joint probability as

$$
\mathbb{P}(X_1, \dots, X_T, Y_1, \dots, Y_T) = \mathbb{P}(X_1) \left(\prod_{t=2}^T \mathbb{P}(X_t | X_{t - 1})\right) \left(\prod_{t=1}^T \mathbb{P}(Y_t | X_t)\right)
$$

Examples can be:
- $X_t$ is the true temperature, $Y_t$ is the observation from a thermostat.
- $X_t$ is the part-of-speech (POS) tag, $Y_t$ is the word in a sentence.

While inference is notoriously expensive for general PGMs, HMM is a tree structure so inference can actually be done in linear time with smart variable elimination.

### Credits
Bryan Chan, Nov. 2024

## Your Task
You are a content creator trying to generate interesting "facts" to get views.
ChatGPT subscription is too expensive so you can only afford to run a small HMM.
Luckily, somewhere on the internet we can find a HMM that has already learned from some text datasets.
The HMM treats $X_t$ as [POS tag](https://en.wikipedia.org/wiki/Part-of-speech_tagging) and $Y_t$ as word.
Given this HMM, we will be generating sentences---one thing to keep in mind is that sentences should follow proper structures---gibberish will not get any likes!

Your task is to:
1. Implement the forward sampling algorithm.
2. Implement the function to compute the joint log probability.
3. Make sure all the functions can be run in linear time!

In [71]:
import _pickle as pickle
import numpy as np

In [72]:
# Load the POS tags and words
(id2word, word2id, id2tag, tag2id) = pickle.load(open("data.pkl", "rb"))

# Load a Hidden Markov Model
(p_init, p_transition, p_emission) = pickle.load(open("hmm.pkl", "rb"))

`p_init` corresponds to $\mathbb{P}(X_0)$.

`p_transition` corresponds to $\mathbb{P}(X_{t+1} | X_{t})$. The row corresponds to the POS tag at current timestep, and the column corresponds to the POS tag at next timestep.

`p_emission` corresponds to $\mathbb{P}(Y_t | X_{t})$. The row corresponds to POS tags, and the column corresponds to words.

Here, note that both `p_transition` and `p_emission` are time homogeneous, meaning that the probability doesn't depend on time, so we can share them across timesteps.
Otherwise, we can expect their shapes to include a time dimension.

In [73]:
p_init.shape, p_transition.shape, p_emission.shape


((36,), (36, 36), (36, 37100))

In [79]:
def sample(T, ref_pos_tag=None):
    """
    Generates a sample of length T from the HMM using the forward sampling algorithm.
    If we provide ref_pos_tag, then we simply generate a sentence that follows the POS tag.
    """
    
    pos_tag = []
    sentence = []
    print(ref_pos_tag)
    # ===============================================================
    # TODO: Implement your code here
    if ref_pos_tag is not None:
        pos_tag.append(ref_pos_tag[0])
        sentence.append(np.random.choice(range(len(p_emission[pos_tag[-1]])), p=p_emission[pos_tag[-1]]))
    else:
        pos_tag.append(np.random.choice(range(len(p_init)), p=p_init))
        sentence.append(np.random.choice(range(len(p_emission[pos_tag[-1]])), p=p_emission[pos_tag[-1]]))
    for t in range(T - 1):
        if ref_pos_tag is not None:
            pos_tag.append(ref_pos_tag[t])
        else:
            pos_tag.append(np.random.choice(range(len(p_init)), p=p_transition[pos_tag[-1]]))
        sentence.append(np.random.choice(range(len(p_emission[pos_tag[-1]])), p=p_emission[pos_tag[-1]]))
    # ===============================================================
    return sentence, pos_tag

In [80]:
def compute_log_prob(sentence, pos_tag):
    """
    Computes the joint log probability of the given sentence and POS tag.
    """

    joint_log_prob = 0

    # ===============================================================
    # TODO: Implement your code here
    
    joint_log_prob += np.log(p_init[pos_tag[0]])
    joint_log_prob += p_emission[pos_tag[0]][sentence[0]]
    for t in range(1, len(pos_tag)):
        joint_log_prob += np.log(p_transition[pos_tag[t-1]][pos_tag[t]])
        joint_log_prob += np.log(p_emission[pos_tag[t]][sentence[t]])
    
    # ===============================================================

    return joint_log_prob

We can try to sample random POS tag and a corresponding sentence

In [81]:
T = 4
sentence, pos_tag = sample(T)

print("Sampled POS tag: {}".format(
    " ".join([id2tag[tag_id] for tag_id in pos_tag])
))

print("Sampled sentence: {}".format(
    " ".join([id2word[word_id] for word_id in sentence])
))

log_prob = compute_log_prob(sentence, pos_tag)
print("Joint log probability: {}".format(log_prob))

None
Sampled POS tag: NNS VBP DT NN
Sampled sentence: DATA ARE A PEACE
Joint log probability: -19.226988135015134


How does the joint log probability compare if we use a proper sentence vs the sampled sentence?

In [82]:
sentence = "I LIKE THE BOOK"
pos_tag = "PRP IN DT NN"

log_prob = compute_log_prob(
    [word2id[word] for word in sentence.split(" ")],
    [tag2id[tag]  for tag in pos_tag.split(" ")]
)
print("Joint log probability: {}".format(log_prob))

Joint log probability: -21.280554546166716


We probably see a lot of non-sense sentences... What if we condition on the POS tag?

In [83]:
T = 4
ref_pos_tag = "PRP IN DT NN"
sentence, pos_tag = sample(
    T,
    [tag2id[tag]  for tag in ref_pos_tag.split(" ")]
)

print("Sampled POS tag: {}".format(
    " ".join([id2tag[tag_id] for tag_id in pos_tag])
))

print("Sampled sentence: {}".format(
    " ".join([id2word[word_id] for word_id in sentence])
))

log_prob = compute_log_prob(sentence, pos_tag)
print("Joint log probability: {}".format(log_prob))

[18, 5, 2, 12]
Sampled POS tag: PRP PRP IN DT
Sampled sentence: THEY OUR OF THE
Joint log probability: -18.621442276975
